<a href="https://colab.research.google.com/github/chukaanyagu-afk/small_potato/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [ ]:
import io
csv_data_string = """longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
-122.23,37.88,41,880,129,322,126,8.3252,452600,NEAR BAY
-122.22,37.86,21,7099,1106,2401,1138,8.3014,358500,NEAR BAY
-122.24,37.85,52,1467,190,496,177,7.2574,352100,NEAR BAY
-122.25,37.85,52,1274,235,558,219,5.6431,341300,NEAR BAY
-122.25,37.85,52,1627,280,565,259,3.8462,342200,NEAR BAY
"""

In [ ]:
housing_data = pd.read_csv(io.StringIO(csv_data_string))



In [ ]:
housing_data.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41,880,129,322,126,8.3252,452600,NEAR BAY
1,-122.22,37.86,21,7099,1106,2401,1138,8.3014,358500,NEAR BAY
2,-122.24,37.85,52,1467,190,496,177,7.2574,352100,NEAR BAY
3,-122.25,37.85,52,1274,235,558,219,5.6431,341300,NEAR BAY
4,-122.25,37.85,52,1627,280,565,259,3.8462,342200,NEAR BAY


In [ ]:
x=housing_data.drop('median_house_value',axis=1)
y=housing_data['median_house_value']

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.30,random_state=2024)

In [ ]:
print(x_train.info())
print(x_train.isna().sum())


<class 'pandas.core.frame.DataFrame'>
Index: 3 entries, 4 to 0
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           3 non-null      float64
 1   latitude            3 non-null      float64
 2   housing_median_age  3 non-null      int64  
 3   total_rooms         3 non-null      int64  
 4   total_bedrooms      3 non-null      int64  
 5   population          3 non-null      int64  
 6   households          3 non-null      int64  
 7   median_income       3 non-null      float64
 8   ocean_proximity     3 non-null      object 
dtypes: float64(3), int64(5), object(1)
memory usage: 240.0+ bytes
None
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
ocean_proximity       0
dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
housing_cat=x_train.select_dtypes(include='object').columns
housing_num=x_train.select_dtypes(exclude='object').columns

In [ ]:
print(housing_cat)
print(housing_num)

Index(['ocean_proximity'], dtype='object')
Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')


In [ ]:
num_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('std_scaler',StandardScaler())
])

In [ ]:
preprocessing=ColumnTransformer([
    ('num',num_pipeline,housing_num),
    ('cat',OneHotEncoder(handle_unknown='ignore'),housing_cat)
])

In [ ]:
check_train=preprocessing.fit_transform(x_train)
transformed_feature_names = preprocessing.get_feature_names_out()
check_train_df = pd.DataFrame(check_train, columns=transformed_feature_names)
check_train_df.head()

,num__longitude,num__latitude,num__housing_median_age,num__total_rooms,num__total_bedrooms,num__population,num__households,num__median_income,cat__ocean_proximity_NEAR BAY
0,-1.224745e+00,-0.707107,0.707107,0.941438,1.295212,1.017167,1.308109,-1.376893,1.0
1,1.740467e-12,-0.707107,0.707107,0.443213,-0.155855,0.342316,-0.188611,0.408939,1.0
2,1.224745e+00,1.414214,-1.414214,-1.384651,-1.139357,-1.359482,-1.119498,0.967954,1.0


In [ ]:
check_train_df=pd.DataFrame(check_train)
check_train_df.isna().sum()

,0
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

In [ ]:
model=Ridge()

In [ ]:
final_pipeline=Pipeline([
    ('preprocessing',preprocessing),
    ('model',model)
])

In [ ]:
final_pipeline.predict(x_train)

array([340076.28942862, 359684.7010384 , 447139.00953297])

In [ ]:
grid=dict()

In [ ]:
import numpy as np
grid['model__alpha'] = np.arange(0.1, 1.1, 0.1)

In [ ]:
search=GridSearchCV(estimator=final_pipeline,param_grid=grid,scoring='neg_mean_absolute_error',cv=3)
results=search.fit(x_train,y_train)
print(results.best_score_)
print(results.best_params_)

-66843.44490747536
{'model__alpha': np.float64(1.0)}


In [ ]:
model = Ridge(alpha=results.best_params_.get('model__alpha'))
final_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', model)
])
final_pipeline.fit(x_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('std_scaler',
                                                                   StandardScaler())]),
                                                  Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['ocean_proximity'], dtype='object'))])),
                ('model', Ridge(alpha=np.float64(1.0)))])